<a href="https://colab.research.google.com/github/yutaota/intro_to_causal_inference/blob/main/Ex09_Synthetic_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 合成コントロール法

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import seaborn as sns

# 合成コントロール法の疑似データ生成と推定プログラム

# データ生成のためのパラメータ設定
np.random.seed(42) # 再現性のためのシード設定
num_treated = 1 # 処置群の地域数
num_control = 30 # 対照群の地域数
num_pre_periods = 20 # 処置前期間数
num_post_periods = 10 # 処置後期間数
tau = 10 # 処置効果 (処置後の処置群に加算される値)

# 総地域数と総期間数
num_areas = num_treated + num_control
num_periods = num_pre_periods + num_post_periods

# データ格納用のリスト
data_list = []

# 地域固定効果 (各地域の固有のベースラインレベル)
# 対照群と処置群で異なる平均レベルを設定
area_effects_control = np.random.normal(loc=50, scale=15, size=num_control)
area_effects_treated = np.random.normal(loc=60, scale=15, size=num_treated) # 処置群は少し高めに設定
area_effects = np.concatenate([area_effects_treated, area_effects_control])

# 時間トレンド (全地域共通のトレンド)
# 例えば、線形トレンド
time_trend = np.linspace(0, 20, num_periods) # 時点経過による増加

# データ生成
area_id_counter = 0
for i in range(num_areas):
    area_id = area_id_counter
    area_id_counter += 1
    is_treated = 1 if i < num_treated else 0 # 最初のnum_treatedが処置群

    for t in range(num_periods):
        is_post = 1 if t >= num_pre_periods else 0 # 処置後期間

        # 基本のアウトカム値: 地域固定効果 + 時間トレンド + ノイズ
        # 地域固定効果に時間トレンドを加えることで、平行トレンドを（ある程度）仮定
        y_base = area_effects[i] + time_trend[t] + np.random.normal(loc=0, scale=2) # ノイズを加える

        # 処置効果: 処置群 (is_treated=1) かつ 処置後 (is_post=1) にのみ加算
        treatment_effect = tau * is_treated * is_post

        # 最終的なアウトカム
        y = y_base + treatment_effect

        data_list.append({
            'area_id': area_id,
            'is_treated': is_treated,
            'period': t,
            'outcome': y
        })

data = pd.DataFrame(data_list) # DataFrameに変換

In [ ]:
# 各系列の時系列プロット

plt.figure(figsize=(12, 15)) # グラフサイズを調整

# 対照群のアウトカムデータを準備
df_treated_only = data[data['is_treated'] == 1].pivot(index='area_id', columns='period', values='outcome')
df_control_only = data[data['is_treated'] == 0].pivot(index='area_id', columns='period', values='outcome')
control_outcomes = df_control_only.values # 各対照地域の時系列データ (num_control x num_periods)

# 対照群の地域数だけループ
colors_control = sns.color_palette(n_colors=num_control)

for i in range(num_control):
    area_id = df_control_only.index[i]
    plt.plot(range(num_periods), control_outcomes[i, :], linestyle='-', color=colors_control[i], alpha=0.5) # 薄い色でプロット

# 処置群の実際のアウトカムをプロット (強調表示)
plt.plot(range(num_periods), df_treated_only.iloc[0].values, marker='o', linestyle='-', color='blue', linewidth=2, label='Treated Area (Actual)')

# # 合成コントロールのアウトカムをプロット (強調表示)
# plt.plot(range(num_periods), synthetic_control_outcome, marker='x', linestyle='--', color='red', linewidth=2, label='Synthetic Control')

# 処置が開始された時点を示す縦線
plt.axvline(num_pre_periods - 0.5, color='gray', linestyle='--', label='Treatment Start')

plt.xlabel('Period')
plt.ylabel('Outcome')
plt.title('Synthetic Control Method: Individual Control Areas vs. Synthetic vs. Treated')
plt.xticks(range(num_periods))
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# --- 合成コントロール法の推定 ---

# 1. 処置群のアウトカムデータを準備
df_treated_only = data[data['is_treated'] == 1].pivot(index='area_id', columns='period', values='outcome')
treated_outcome = df_treated_only.iloc[0].values # 処置群は1つなので、そのアウトカムを取得

# 2. 対照群のアウトカムデータを準備
df_control_only = data[data['is_treated'] == 0].pivot(index='area_id', columns='period', values='outcome')
control_outcomes = df_control_only.values # 各対照地域の時系列データ (num_control x num_periods)

# 3. 合成コントロールに使用する共変量（処置前期間のアウトカム）を準備
# 処置前期間のアウトカムを共変量と見なす
X_pre_control = control_outcomes[:, :num_pre_periods] # 対照群の処置前期間のアウトカム (num_control x num_pre_periods)
y_pre_treated = treated_outcome[:num_pre_periods].reshape(-1, 1) # 処置群の処置前期間のアウトカム (num_pre_periods x 1)

# 4. 制約付き最小二乗法で合成コントロールの重み (W) を推定
# Wは以下の条件を満たすように推定される：
#   - X_pre_control.T @ W が y_pre_treated に近くなるように最小二乗法で推定
#   - 重み W の要素は非負 (w_j >= 0 for all j)
#   - 重みの合計が1 (sum(w_j) = 1)

# 目的関数: ||X_pre_control @ W - y_pre_treated||^2 の最小化 (Wについて)
# 制約: W >= 0, sum(W) = 1

# SciPyの最適化ライブラリを使用する
from scipy.optimize import minimize

# 目的関数 (最小化したい関数)
# minimizeは引数を単一のベクトルとして受け取るため、Wを1次元ベクトルとして扱う
def objective_function(W, X, y):
    return np.sum((X.T @ W - y.flatten())**2) # yをflattenして1次元にする

# 制約条件
# 線形制約の場合、以下の形式で指定: A_eq @ W - b_eq = 0 または A_ineq @ W - b_ineq >= 0
# sum(W) = 1  => [1, 1, ..., 1] @ W - 1 = 0
constraints = ({'type': 'eq', 'fun': lambda W: np.sum(W) - 1})

# 境界条件 (重みは非負)
# 各重み W_j に対して 0 <= W_j <= None (上限なし)
bounds = [(0, None) for _ in range(num_control)]

# 初期値 (例えば、すべての対照群に均等な重みを与える)
initial_weights = np.ones(num_control) / num_control

# 最適化を実行
# method='SLSQP'は、等式・不等式制約と境界条件を扱える手法
result = minimize(
    objective_function,
    initial_weights,
    args=(X_pre_control, y_pre_treated),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

estimated_weights = result.x
print("\n合成コントロールの重み (W):")
print([round(x, 3) for x in estimated_weights.tolist()])
# 重みの合計を確認 (1に近いか)
print(f"重みの合計: {np.sum(estimated_weights):.4f}")

# 5. 合成コントロールの時系列データを生成
# 推定された重みを使って、全期間の合成コントロールのアウトカムを計算
synthetic_control_outcome = control_outcomes.T @ estimated_weights

# 6. 推定された処置効果を計算
# 処置効果は、処置群の実際のアウトカムと合成コントロールのアウトカムの差
estimated_treatment_effect = treated_outcome - synthetic_control_outcome

# 処置後期間における平均処置効果 (ATT)
att_post_periods = np.mean(estimated_treatment_effect[num_pre_periods:])
print(f"\n処置後期間 ({num_pre_periods}期～{num_periods-1}期) の平均処置効果 (ATT): {att_post_periods:.4f}")
print(f"真の処置効果 (tau): {tau}")

In [ ]:
# 7. 結果のプロット
plt.figure(figsize=(12, 7))

# 処置群の実際のアウトカムをプロット
plt.plot(range(num_periods), treated_outcome, marker='o', linestyle='-', color='blue', label='Treated Area (Actual)')

# 合成コントロールのアウトカムをプロット
plt.plot(range(num_periods), synthetic_control_outcome, marker='x', linestyle='--', color='red', label='Synthetic Control')

# 処置が開始された時点を示す縦線
plt.axvline(num_pre_periods - 0.5, color='gray', linestyle='--', label='Treatment Start')

plt.xlabel('Period')
plt.ylabel('Outcome')
plt.title('Synthetic Control Method: Actual vs. Synthetic Outcome')
plt.xticks(range(num_periods))
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 推定された処置効果のプロット
plt.figure(figsize=(12, 5))
plt.plot(range(num_periods), estimated_treatment_effect, marker='o', linestyle='-', color='green')

# 処置が開始された時点を示す縦線
plt.axvline(num_pre_periods - 0.5, color='gray', linestyle='--', label='Treatment Start')

# 処置効果ゼロの基準線
plt.axhline(0, color='black', linestyle=':', alpha=0.7)

plt.xlabel('Period')
plt.ylabel('Estimated Treatment Effect')
plt.title('Estimated Treatment Effect Over Time (Treated - Synthetic)')
plt.xticks(range(num_periods))
plt.grid(True)
plt.legend()
plt.show()